# ☀️ Les prediccions de taques solars d'en Toni

Aquesta aplicació interactiva permet explorar l'activitat solar històrica i predir la seva evolució utilitzant models d'aprenentatge automàtic d'última generació.

### 📡 Les dades
Les dades són proporcionades pel centre **SILSO (Sunspot Index and Long-term Solar Observations)** a Bèlgica. Podeu consultar les dades originals i la seva metodologia a la seva web oficial:
[www.sidc.be/silso/datafiles](https://www.sidc.be/silso/datafiles)

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
import os
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from datetime import datetime

# Ensure we are in the root directory
if os.path.basename(os.getcwd()) == 'notebooks':
    sys.path.append(os.path.abspath('..'))
    os.chdir('..')

try:
    import gradio as gr
except ImportError:
    !pip install gradio
    import gradio as gr

from src.train import run_future_forecast
from src.utils import load_config

# Load pre-saved data and config
try:
    df_filtered = joblib.load('models/sunspots_data.joblib')
    config = joblib.load('models/config.joblib')
    validation_results = joblib.load('models/validation_results.joblib')
    print("Models i dades carregats correctament.")
except FileNotFoundError:
    print("⚠️ Atenció: No s'han trobat els fitxers de model. Executa primer el notebook d'anàlisi.")
    df_filtered = None
    validation_results = None
    config = None

def interpret_results(avg_val, last_val):
    trend = "estable" if abs(avg_val - last_val) < 5 else ("ascendent" if avg_val > last_val else "descendent")
    activity = "alta" if avg_val > 150 else ("mitjana" if avg_val > 50 else "baixa")
    return f"Es preveu una activitat solar **{activity}** amb una tendència **{trend}** respecte a l'última observació."

def predict_sunspots(steps):
    if df_filtered is None:
        return "Error: Exporta les dades primer.", None, "N/A"
    
    future_predictions = run_future_forecast(df_filtered, steps=int(steps), config=config)
    last_val = df_filtered['SUNSPOTS'].iloc[-1]
    avg_val = np.mean(list(future_predictions.values()))
    
    output_text = "| Data | Predicció |\n| :--- | :--- |\n"
    for date, val in future_predictions.items():
        output_text += f"| {date.strftime('%d/%m/%Y')} | {val:.2f} |\n"
    
    interpretation = interpret_results(avg_val, last_val)
    
    plt.figure(figsize=(12, 5))
    last_60 = df_filtered.tail(60)
    plt.plot(last_60.index, last_60['SUNSPOTS'], label='Real (Darrers 60 dies)', color='#1f77b4', linewidth=2)
    
    pred_dates = list(future_predictions.keys())
    pred_vals = list(future_predictions.values())
    plt.plot(pred_dates, pred_vals, label='Predicció', color='#d62728', marker='o', markersize=6, linestyle='--')
    
    plt.title("Predicció del Nombre de Taques Solars", fontsize=14, pad=15)
    plt.xlabel("Data", fontsize=12)
    plt.ylabel("Nombre de Taques", fontsize=12)
    plt.legend()
    plt.grid(True, alpha=0.2)
    plt.tight_layout()
    
    return output_text, plt, interpretation

def show_validation():
    if validation_results is None:
        return None, "No hi ha dades de validació disponibles."
    
    plt.figure(figsize=(12, 5))
    actuals = validation_results['actuals']
    preds = validation_results['predictions']
    
    # Zoom in on the last 500 points for clarity
    show_n = min(500, len(actuals))
    plt.plot(actuals[-show_n:], label='Real', color='#1f77b4', alpha=0.6)
    plt.plot(preds[-show_n:], label='Model (Hybrid)', color='#d62728', alpha=0.8, linestyle='--')
    
    plt.title("Rendiment Històric (Darreres 500 observacions de test)", fontsize=14)
    plt.legend()
    plt.grid(True, alpha=0.2)
    
    metrics = f"**RMSE:** {validation_results['hybrid_rmse']:.2f} | **MAE:** {validation_results['hybrid_mae']:.2f}"
    return plt, metrics

with gr.Blocks(title="Toni's Sunspot Predictor") as demo:
    gr.Markdown("# ☀️ Taques Solars d'en Toni")
    gr.Markdown("Consulta les dades en temps real de [SILSO](https://www.sidc.be/silso/datafiles)")
    
    with gr.Tabs():
        with gr.Tab("🚀 Predicció Futura"):
            with gr.Row():
                with gr.Column(scale=1):
                    input_steps = gr.Slider(minimum=1, maximum=30, value=7, step=1, label="Dies a predir")
                    btn = gr.Button("Generar Predicció", variant="primary")
                    out_interpret = gr.Markdown("Fes clic per veure l'interpretació.")
                    out_table = gr.Markdown()
                with gr.Column(scale=2):
                    out_plot = gr.Plot()
            
            btn.click(predict_sunspots, inputs=input_steps, outputs=[out_table, out_plot, out_interpret])
        
        with gr.Tab("📊 Rendiment del Model"):
            gr.Markdown("Aquí es mostra com es compara el model amb dades reals que ja coneixem (Backtesting).")
            val_btn = gr.Button("Mostrar Gràfica de Validació")
            val_metrics = gr.Markdown()
            val_plot = gr.Plot()
            val_btn.click(show_validation, outputs=[val_plot, val_metrics])
            
        with gr.Tab("🧠 Com funciona?"):
            gr.Markdown("""
            ### Arquitectura Híbrida
            El model utilitza tres capes per predir el comportament solar:
            
            1. **Ridge Regression (Base Lineal)**: Captura la tendència general i els cicles solars de 11 anys. És robust i evita overfitting.
            2. **XGBoost (Residus No-Lineals)**: Aprèn dels errors de la Ridge. Està especialitzat en fluctuacions de curt termini i dinàmiques complexes que una línia no pot descriure.
            3. **Extreme Value Theory (EVT)**: Serveix per calibrar els extrems. Les taques solars poden tenir pics sobtats (pous o explosions) que la regressió normal sovint ignora.
            
            **Per què fem servir el Random State 7?**
            Perquè és el número de la sort d'en Toni i assegura que si algú altre repeteix l'experiment, obtindrà exactament els mateixos resultats.
            """)

if __name__ == "__main__":